In [ ]:
class Character:
    def __init__(self, name, role):
        self.name = name
        self.role = role


class PlayerState:
    def __init__(self, name):
        self.name = name
        self.energy = 3
        self.reputation = 0
        self.courage = 0
        self.inventory = []
        self.clues = set()
        self.route = []
        self.ending = ''


class CampNovel:
    def __init__(self):
        self.characters = [
            Character('Алиса', 'лидер отряда'),
            Character('Электроник', 'техник'),
            Character('Ольга Дмитриевна', 'вожатая')
        ]

        self.locations = {
            'площадь': 'У памятника кто-то оставил следы от ботинок.',
            'сцена': 'За кулисами найдена записка с временем встречи.',
            'столовая': 'Дежурные слышали шум у склада после отбоя.',
            'лодочная': 'Под настилом найден ржавый ключ.',
            'радиорубка': 'В журнале дежурств вырвана страница.'
        }

        self.roles_info = {
            'лидер отряда': 'координирует смену и знает всех ребят',
            'техник': 'работает с аппаратурой и ключами',
            'вожатая': 'следит за дисциплиной и безопасностью'
        }

        self.state = None

    def custom_print(self, text):
        print(text)

    def ask(self, prompt):
        self.custom_print(prompt)
        return input('> ').strip()

    def choose(self, prompt, options):
        self.custom_print('')
        self.custom_print(prompt)
        for key in options:
            self.custom_print(str(key) + '. ' + options[key])
        answer = input('> ').strip()
        while answer not in options:
            self.custom_print('Нужно ввести один из номеров.')
            answer = input('> ').strip()
        return answer

    def intro(self):
        name = self.ask('Введите имя героя:')
        if name == '':
            name = 'Семён'
        self.state = PlayerState(name)

        self.custom_print('')
        self.custom_print('Лагерь "Солнечная Заря". Ночью пропала архивная папка с документами.')
        self.custom_print('Тебе нужно провести маленькое расследование до общего сбора.')
        self.state.route.append('Прибытие в лагерь')

        self.custom_print('')
        self.custom_print('Персонажи смены:')
        for c in self.characters:
            self.custom_print('- ' + c.name + ' (' + c.role + ': ' + self.roles_info[c.role] + ')')

    def first_choice(self):
        ans = self.choose(
            'Первое решение:',
            {'1': 'Помочь вожатой с опросом', '2': 'Идти в одиночный поиск'}
        )

        if ans == '1':
            self.state.reputation += 2
            self.state.inventory.append('блокнот вожатой')
            self.state.route.append('Помощь вожатой')
            self.custom_print('Ты помог(ла) вожатой и получил(а) блокнот с заметками.')
        else:
            self.state.courage += 2
            self.state.energy -= 1
            self.state.inventory.append('карта лагеря')
            self.state.route.append('Одиночный поиск')
            self.custom_print('Ты выбрал(а) самостоятельный поиск и нашел(ла) карту лагеря.')

    def inspect_locations(self):
        self.custom_print('')
        self.custom_print('До отбоя можно проверить только 3 локации.')
        checked = 0
        for place in self.locations:
            if checked == 3:
                break

            ans = self.choose(
                'Осмотреть локацию ' + place + '?',
                {'1': 'Да', '2': 'Пропустить'}
            )

            if ans == '1':
                checked += 1
                self.state.route.append('Осмотр: ' + place)
                self.state.clues.add(place)
                self.state.inventory.append('улика: ' + place)
                self.custom_print(self.locations[place])

                if place == 'столовая':
                    self.state.reputation += 1
                if place == 'лодочная':
                    self.state.courage += 1
            else:
                self.state.route.append('Пропуск: ' + place)

        if checked == 0:
            self.state.energy += 1
            self.custom_print('Ты ничего не проверил(а), но успел(а) немного отдохнуть.')

    def radio_stage(self):
        self.custom_print('')
        self.custom_print('Электроник говорит: в радиорубке есть код доступа к архиву.')
        attempts = 3
        while attempts > 0:
            code = self.ask('Введи 4-значный код:')
            if code == '1989':
                self.state.reputation += 1
                self.state.clues.add('код-1989')
                self.state.route.append('Радиорубка взломана')
                self.custom_print('Код верный. В журнале найдено имя ночного дежурного.')
                return True
            attempts -= 1
            self.custom_print('Код неверный. Осталось попыток: ' + str(attempts))

        self.state.route.append('Радиорубка не взломана')
        return False

    def night_stage(self):
        ans = self.choose(
            'Ночью у склада мелькнул силуэт. Что делать?',
            {'1': 'Преследовать', '2': 'Позвать помощь'}
        )

        if ans == '1':
            self.state.courage += 1
            self.state.energy -= 1
            self.state.clues.add('следы у склада')
            self.state.route.append('Ночное преследование')
            self.custom_print('Ты побежал(а) за силуэтом и нашел(ла) новые следы.')
        else:
            self.state.reputation += 1
            self.state.route.append('Позвал(а) помощь')
            self.custom_print('Ты поднял(а) отряд, и поиск стал безопаснее.')

    def optimize_inventory(self):
        if len(self.state.inventory) > 6:
            self.state.inventory.pop(0)

        if 'карта лагеря' in self.state.inventory and 'блокнот вожатой' in self.state.inventory:
            self.state.inventory.remove('карта лагеря')
            self.state.inventory.append('карта с пометками')

        if 'улика: сцена' in self.state.inventory:
            self.state.clues.add('переписанный сценарий')

    def final_stage(self, radio_success):
        accuse = self.choose(
            'Финальный сбор: кого обвинить?',
            {'1': 'Алису', '2': 'Электроника', '3': 'Никого (показать факты)'}
        )

        if accuse == '1':
            self.state.route.append('Обвинение Алисы')
        elif accuse == '2':
            self.state.route.append('Обвинение Электроника')
        else:
            self.state.reputation += 1
            self.state.route.append('Без обвинений, анализ фактов')

        many_clues = len(self.state.clues) >= 3
        trusted = self.state.reputation >= 3
        brave = self.state.courage >= 2

        if radio_success and many_clues and trusted and brave and accuse == '3':
            self.state.ending = 'Истинная концовка: ты раскрыл(а) подмену архивов и спас(ла) смену.'
        elif many_clues and accuse in ('1', '2'):
            self.state.ending = 'Драматичная концовка: виновный назван, но лагерь расколот на два лагеря.'
        elif radio_success or trusted:
            self.state.ending = 'Нейтральная концовка: часть правды найдена, но не вся история раскрыта.'
        else:
            self.state.ending = 'Плохая концовка: доказательств не хватило, дело закрыли без ответа.'

    def build_report(self):
        report = {}
        report['герой'] = self.state.name
        report['энергия'] = self.state.energy
        report['репутация'] = self.state.reputation
        report['смелость'] = self.state.courage
        report['маршрут'] = self.state.route
        report['улики'] = list(self.state.clues)
        report['инвентарь'] = self.state.inventory
        report['концовка'] = self.state.ending
        return report

    def save_report(self, report):
        lines = []
        lines.append('Герой: ' + report['герой'])
        lines.append('Энергия: ' + str(report['энергия']))
        lines.append('Репутация: ' + str(report['репутация']))
        lines.append('Смелость: ' + str(report['смелость']))
        lines.append('Маршрут:')
        for step in report['маршрут']:
            lines.append('- ' + step)
        lines.append('Улики: ' + ', '.join(report['улики']))
        lines.append('Инвентарь: ' + ', '.join(report['инвентарь']))
        lines.append('Финал: ' + report['концовка'])

        with open('novel_result.txt', 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))

    def run(self):
        self.intro()
        self.first_choice()
        self.inspect_locations()
        radio_success = self.radio_stage()
        self.night_stage()
        self.optimize_inventory()
        self.final_stage(radio_success)

        report = self.build_report()
        self.save_report(report)

        self.custom_print('')
        self.custom_print(self.state.ending)
        self.custom_print('Итог прохождения сохранен в novel_result.txt')


game = CampNovel()
game.run()
